# train_architecture_tests_v1

Notebook for architecture experiments and error analysis.

Included:
- single-head baseline per domain (`Moscow`, `Regions`, optional `Genplan`)
- hierarchical pipeline (`l1 -> l2`) per domain
- domain router + domain heads (`l2`)
- export of misclassified rows for manual review


In [1]:
import gc
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

RANDOM_STATE = 44


In [ ]:
BASE = Path(r"C:/Users/artem/MAIN/projects/plain/local/python_modules/scripts/geo")
OUT_DIR = BASE / "reports" / "arch_tests"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "Moscow": BASE / "labeled_msk_df.csv",
    "Regions": BASE / "labeled_regions_df.csv",
    "Regions_fallback": BASE / "labeled_regions_df.csv",
    "Genplan": BASE / "labeled_genplan_df.csv",
}

USECOLS = ["Source_File", "Layer", "Type", "Text", "l1", "l2"]

QUICK_MODE = True
QUICK_GROUPS = {
    "Moscow": 70,
    "Regions": 15,
    "Genplan": 20,
}

MEMORY_SAFE_MODE = True
TFIDF_DTYPE = np.float32 if MEMORY_SAFE_MODE else np.float64

SGD_CHAR_MAX_FEATURES = 120_000 if MEMORY_SAFE_MODE else 220_000
SVC_CHAR_MAX_FEATURES = 140_000 if MEMORY_SAFE_MODE else 260_000
UNION_CHAR_MAX_FEATURES = 80_000 if MEMORY_SAFE_MODE else 180_000
UNION_WORD_MAX_FEATURES = 25_000 if MEMORY_SAFE_MODE else 80_000

TEST_SIZE = 0.2
VAL_SIZE_IN_TV = 0.2

RARE_THRESHOLD_L1_BY_DOMAIN = {
    "Moscow": None,
    "Regions": None,
    "Genplan": None,
}
RARE_THRESHOLD_L2_BY_DOMAIN = {
    "Moscow": 30,
    "Regions": 100,
    "Genplan": 50,
}

BASE_MODEL_KIND = "linear_char"
RUN_CLASSIFICATION_REPORT = False


In [3]:
def resolve_regions_path() -> Path:
    return PATHS["Regions"] if PATHS["Regions"].exists() else PATHS["Regions_fallback"]


def load_domain_df(name: str, path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"{name}: file not found -> {path}")

    df = pd.read_csv(path, usecols=USECOLS, low_memory=False)
    for col in ["Source_File", "Layer", "Type", "Text"]:
        df[col] = df[col].fillna("").astype(str)
    df["domain"] = name

    print(f"Loaded {name:<8} | rows={len(df):,} | files={df['Source_File'].nunique():,} | path={path.name}")
    return df


def maybe_quick_subset(df: pd.DataFrame, domain: str) -> pd.DataFrame:
    if not QUICK_MODE:
        return df

    n_groups = QUICK_GROUPS.get(domain)
    if n_groups is None:
        return df

    groups = pd.Series(df["Source_File"].unique())
    if n_groups >= len(groups):
        return df

    keep = groups.sample(n=n_groups, random_state=RANDOM_STATE)
    out = df[df["Source_File"].isin(keep)].copy()
    print(f"{domain:<8} quick subset -> rows={len(out):,}, files={out['Source_File'].nunique():,}")
    return out


datasets = {}
datasets["Moscow"] = maybe_quick_subset(load_domain_df("Moscow", PATHS["Moscow"]), "Moscow")
datasets["Regions"] = maybe_quick_subset(load_domain_df("Regions", resolve_regions_path()), "Regions")

if PATHS["Genplan"].exists():
    datasets["Genplan"] = maybe_quick_subset(load_domain_df("Genplan", PATHS["Genplan"]), "Genplan")

print("Domains in run:", list(datasets.keys()))


Loaded Moscow   | rows=180,953 | files=370 | path=labeled_msk_df.csv
Moscow   quick subset -> rows=70,014, files=70
Loaded Regions  | rows=3,069,680 | files=146 | path=labeled_regions_df.csv
Regions  quick subset -> rows=384,457, files=15
Domains in run: ['Moscow', 'Regions']


In [4]:
def build_text_features(df: pd.DataFrame) -> pd.Series:
    return (
        "TYPE=" + df["Type"].astype(str) + " " +
        "LAYER=" + df["Layer"].astype(str) + " " +
        "TEXT=" + df["Text"].astype(str)
    )


def clean_label(df: pd.DataFrame, label_col: str) -> pd.Series:
    return df[label_col].fillna("").astype(str).str.strip()


def filter_non_empty(df: pd.DataFrame, label_col: str) -> pd.DataFrame:
    y = clean_label(df, label_col)
    return df[y != ""].copy()


class RareLabelMapper:
    def __init__(self, threshold=None, unknown_label="unknown"):
        self.threshold = threshold
        self.unknown_label = unknown_label
        self.rare_labels_ = set()

    def fit(self, y: pd.Series):
        if self.threshold is None:
            self.rare_labels_ = set()
            return self
        counts = y.value_counts()
        self.rare_labels_ = set(counts[counts < self.threshold].index)
        return self

    def transform(self, y: pd.Series) -> pd.Series:
        if not self.rare_labels_:
            return y
        return y.where(~y.isin(self.rare_labels_), self.unknown_label)

    def fit_transform(self, y: pd.Series) -> pd.Series:
        return self.fit(y).transform(y)


def split_group_holdout(X, y, groups, test_size=TEST_SIZE, val_size_in_tv=VAL_SIZE_IN_TV, random_state=RANDOM_STATE):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    tv_idx, te_idx = next(gss.split(X, y, groups=groups))

    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size_in_tv, random_state=random_state)
    tr_rel, va_rel = next(gss2.split(X.iloc[tv_idx], y.iloc[tv_idx], groups=groups.iloc[tv_idx]))

    tr_idx = X.iloc[tv_idx].iloc[tr_rel].index
    va_idx = X.iloc[tv_idx].iloc[va_rel].index
    te_idx = X.iloc[te_idx].index
    return tr_idx, va_idx, te_idx


In [5]:
def make_sgd_char():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=SGD_CHAR_MAX_FEATURES,
            dtype=TFIDF_DTYPE,
        )),
        ("clf", SGDClassifier(
            loss="log_loss",
            alpha=1e-5,
            max_iter=50,
            tol=1e-3,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        )),
    ])


def make_linear_svc_char(C=0.5):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=SVC_CHAR_MAX_FEATURES,
            dtype=TFIDF_DTYPE,
        )),
        ("clf", LinearSVC(
            C=C,
            max_iter=12000,
            class_weight="balanced",
            dual=False,
        )),
    ])


def make_linear_svc_union(C=0.5):
    return Pipeline([
        ("tfidf", FeatureUnion([
            ("char", TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                min_df=2,
                max_features=UNION_CHAR_MAX_FEATURES,
                dtype=TFIDF_DTYPE,
            )),
            ("word", TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                min_df=2,
                max_features=UNION_WORD_MAX_FEATURES,
                dtype=TFIDF_DTYPE,
            )),
        ])),
        ("clf", LinearSVC(
            C=C,
            max_iter=12000,
            class_weight="balanced",
            dual=False,
        )),
    ])


def make_base_model(kind=BASE_MODEL_KIND):
    if kind == "sgd_char":
        return make_sgd_char()
    if kind == "linear_union":
        return make_linear_svc_union(C=0.5)
    return make_linear_svc_char(C=0.5)


In [6]:
def safe_decision_margin(model, X):
    # Margin can help prioritize uncertain errors for manual review.
    if not hasattr(model, "decision_function"):
        return np.full(shape=(len(X),), fill_value=np.nan)

    try:
        scores = model.decision_function(X)
    except Exception:
        return np.full(shape=(len(X),), fill_value=np.nan)

    if isinstance(scores, list):
        scores = np.asarray(scores)

    if getattr(scores, "ndim", 1) == 1:
        return np.abs(scores)

    if scores.shape[1] == 1:
        return np.abs(scores[:, 0])

    top2 = np.partition(scores, kth=-2, axis=1)[:, -2:]
    return top2[:, 1] - top2[:, 0]


def metrics_dict(y_true, y_pred):
    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "n_rows": len(y_true),
    }


def summarize_confusions(y_true, y_pred, top_k=20):
    cm = pd.crosstab(pd.Series(y_true, name="y_true"), pd.Series(y_pred, name="y_pred"))
    rows = []
    for true_label in cm.index:
        for pred_label in cm.columns:
            if true_label == pred_label:
                continue
            cnt = int(cm.loc[true_label, pred_label])
            if cnt > 0:
                rows.append((true_label, pred_label, cnt))

    out = pd.DataFrame(rows, columns=["y_true", "y_pred", "count"])
    if out.empty:
        return out
    return out.sort_values("count", ascending=False).head(top_k)


def export_errors(df: pd.DataFrame, name: str) -> Path:
    ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    path = OUT_DIR / f"{name}_errors_{ts}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved: {path}")
    return path


In [7]:
@dataclass
class PreparedTask:
    domain: str
    label_col: str
    X_tr: pd.Series
    X_va: pd.Series
    X_te: pd.Series
    y_tr: pd.Series
    y_va: pd.Series
    y_te: pd.Series
    meta_te: pd.DataFrame
    rare_mapper: RareLabelMapper


def prepare_task(df: pd.DataFrame, domain: str, label_col: str, rare_threshold=None) -> PreparedTask:
    work = filter_non_empty(df, label_col)

    X = build_text_features(work)
    y_raw = clean_label(work, label_col)
    groups = work["Source_File"].astype(str)

    tr_idx, va_idx, te_idx = split_group_holdout(X, y_raw, groups)

    y_tr_raw = y_raw.loc[tr_idx]
    y_va_raw = y_raw.loc[va_idx]
    y_te_raw = y_raw.loc[te_idx]

    mapper = RareLabelMapper(threshold=rare_threshold)
    y_tr = mapper.fit_transform(y_tr_raw)
    y_va = mapper.transform(y_va_raw)
    y_te = mapper.transform(y_te_raw)

    meta_cols = ["domain", "Source_File", "Layer", "Type", "Text", "l1", "l2"]
    meta_te = work.loc[te_idx, meta_cols].copy()

    return PreparedTask(
        domain=domain,
        label_col=label_col,
        X_tr=X.loc[tr_idx],
        X_va=X.loc[va_idx],
        X_te=X.loc[te_idx],
        y_tr=y_tr,
        y_va=y_va,
        y_te=y_te,
        meta_te=meta_te,
        rare_mapper=mapper,
    )


def run_single_head(task: PreparedTask, model_name: str, model_factory):
    model = model_factory()
    model.fit(task.X_tr, task.y_tr)

    val_pred = pd.Series(model.predict(task.X_va), index=task.X_va.index)
    te_pred = pd.Series(model.predict(task.X_te), index=task.X_te.index)

    metrics = []
    for split, yy, pp in [
        ("val", task.y_va, val_pred),
        ("test", task.y_te, te_pred),
    ]:
        row = metrics_dict(yy, pp)
        row.update({
            "arch": "single_head",
            "model": model_name,
            "domain": task.domain,
            "label": task.label_col,
            "split": split,
            "classes_train": task.y_tr.nunique(),
            "rare_labels_merged": len(task.rare_mapper.rare_labels_),
        })
        metrics.append(row)

    margin = safe_decision_margin(model, task.X_te)
    err = task.meta_te.copy()
    err["label"] = task.label_col
    err["arch"] = "single_head"
    err["model"] = model_name
    err["y_true"] = task.y_te.values
    err["y_pred"] = te_pred.values
    err["is_error"] = err["y_true"] != err["y_pred"]
    err["margin"] = margin

    return pd.DataFrame(metrics), err, model


## 1) Single-head baseline per domain


In [8]:
single_rows = []
single_errors = []

for domain, ddf in datasets.items():
    for label_col in ["l1", "l2"]:
        rare_thr = RARE_THRESHOLD_L1_BY_DOMAIN.get(domain) if label_col == "l1" else RARE_THRESHOLD_L2_BY_DOMAIN.get(domain)
        task = prepare_task(ddf, domain=domain, label_col=label_col, rare_threshold=rare_thr)

        m, err, _ = run_single_head(
            task,
            model_name=BASE_MODEL_KIND,
            model_factory=lambda: make_base_model(BASE_MODEL_KIND),
        )
        single_rows.append(m)
        single_errors.append(err)
        gc.collect()

single_metrics = pd.concat(single_rows, ignore_index=True)
single_errors = pd.concat(single_errors, ignore_index=True)

display(single_metrics.sort_values(["domain", "label", "split", "macro_f1"], ascending=[True, True, True, False]))


,macro_f1,weighted_f1,accuracy,n_rows,arch,model,domain,label,split,classes_train,rare_labels_merged
1,0.879848,0.989054,0.987097,2015,single_head,linear_char,Moscow,l1,test,12,0
0,0.990344,0.997978,0.997994,3489,single_head,linear_char,Moscow,l1,val,12,0
3,0.939086,0.988250,0.988975,1542,single_head,linear_char,Moscow,l2,test,24,8
2,0.994475,0.995445,0.995375,2811,single_head,linear_char,Moscow,l2,val,24,8
5,0.666363,0.900051,0.905374,173145,single_head,linear_char,Regions,l1,test,10,0
4,0.791688,0.756565,0.774263,27944,single_head,linear_char,Regions,l1,val,10,0
7,0.657530,0.921735,0.907465,173145,single_head,linear_char,Regions,l2,test,20,17
6,0.702389,0.857050,0.838999,27944,single_head,linear_char,Regions,l2,val,20,17


In [9]:
single_test = single_metrics[single_metrics["split"] == "test"].copy()
for c in ["macro_f1", "weighted_f1", "accuracy"]:
    single_test[c] = single_test[c].round(4)

display(single_test[[
    "domain", "label", "arch", "model", "macro_f1", "weighted_f1", "accuracy", "classes_train", "rare_labels_merged", "n_rows"
]].sort_values(["domain", "label", "macro_f1"], ascending=[True, True, False]))


,domain,label,arch,model,macro_f1,weighted_f1,accuracy,classes_train,rare_labels_merged,n_rows
1,Moscow,l1,single_head,linear_char,0.8798,0.9891,0.9871,12,0,2015
3,Moscow,l2,single_head,linear_char,0.9391,0.9882,0.9890,24,8,1542
5,Regions,l1,single_head,linear_char,0.6664,0.9001,0.9054,10,0,173145
7,Regions,l2,single_head,linear_char,0.6575,0.9217,0.9075,20,17,173145


## 2) Hierarchical architecture (`l1 -> l2`) per domain

Vectorized inference version (batch by predicted `l1`).


In [10]:
def run_hierarchical_domain(df: pd.DataFrame, domain: str, model_factory, rare_threshold_l2=None, min_local_support=200):
    work = df.copy()
    y1 = clean_label(work, "l1")
    y2 = clean_label(work, "l2")
    mask = (y1 != "") & (y2 != "")
    work = work[mask].copy()

    X = build_text_features(work)
    y1 = clean_label(work, "l1")
    y2_raw = clean_label(work, "l2")
    groups = work["Source_File"].astype(str)

    tr_idx, va_idx, te_idx = split_group_holdout(X, y1, groups)

    X_tr, X_va, X_te = X.loc[tr_idx], X.loc[va_idx], X.loc[te_idx]
    y1_tr, y1_va, y1_te = y1.loc[tr_idx], y1.loc[va_idx], y1.loc[te_idx]

    y2_tr_raw, y2_va_raw, y2_te_raw = y2_raw.loc[tr_idx], y2_raw.loc[va_idx], y2_raw.loc[te_idx]
    mapper = RareLabelMapper(threshold=rare_threshold_l2)
    y2_tr = mapper.fit_transform(y2_tr_raw)
    y2_va = mapper.transform(y2_va_raw)
    y2_te = mapper.transform(y2_te_raw)

    l1_model = model_factory()
    l1_model.fit(X_tr, y1_tr)

    l2_global = model_factory()
    l2_global.fit(X_tr, y2_tr)

    local_models = {}
    counts = y1_tr.value_counts()
    for cls, cnt in counts.items():
        if cnt < min_local_support:
            continue
        idx = y1_tr[y1_tr == cls].index
        y2_sub = y2_tr.loc[idx]
        if y2_sub.nunique() < 2:
            continue
        mdl = model_factory()
        mdl.fit(X_tr.loc[idx], y2_sub)
        local_models[cls] = mdl

    def predict_l2_by_l1(xx):
        p1 = pd.Series(l1_model.predict(xx), index=xx.index)
        p2 = pd.Series(index=xx.index, dtype=object)

        # Global fallback first, then overwrite where local model exists.
        p2.loc[:] = l2_global.predict(xx)

        for cls, mdl in local_models.items():
            idx = p1[p1 == cls].index
            if len(idx) == 0:
                continue
            p2.loc[idx] = mdl.predict(xx.loc[idx])
        return p1, p2

    rows = []
    err_rows = []
    for split_name, xx, yy1, yy2 in [
        ("val", X_va, y1_va, y2_va),
        ("test", X_te, y1_te, y2_te),
    ]:
        p1, p2 = predict_l2_by_l1(xx)

        row1 = metrics_dict(yy1, p1)
        row1.update({
            "arch": "hierarchical",
            "model": BASE_MODEL_KIND,
            "domain": domain,
            "label": "l1",
            "split": split_name,
            "classes_train": y1_tr.nunique(),
            "rare_labels_merged": 0,
            "local_l2_models": len(local_models),
        })

        row2 = metrics_dict(yy2, p2)
        row2.update({
            "arch": "hierarchical",
            "model": BASE_MODEL_KIND,
            "domain": domain,
            "label": "l2",
            "split": split_name,
            "classes_train": y2_tr.nunique(),
            "rare_labels_merged": len(mapper.rare_labels_),
            "local_l2_models": len(local_models),
        })
        rows.extend([row1, row2])

        if split_name == "test":
            meta = work.loc[te_idx, ["domain", "Source_File", "Layer", "Type", "Text", "l1", "l2"]].copy()
            e = meta.copy()
            e["arch"] = "hierarchical"
            e["model"] = BASE_MODEL_KIND
            e["label"] = "l2"
            e["y_true"] = yy2.values
            e["y_pred"] = p2.values
            e["y1_true"] = yy1.values
            e["y1_pred"] = p1.values
            e["is_error"] = e["y_true"] != e["y_pred"]
            e["is_l1_error"] = e["y1_true"] != e["y1_pred"]
            err_rows.append(e)

    return pd.DataFrame(rows), (pd.concat(err_rows, ignore_index=True) if err_rows else pd.DataFrame())


In [11]:
hier_rows = []
hier_errors = []

for domain, ddf in datasets.items():
    rare_thr_l2 = RARE_THRESHOLD_L2_BY_DOMAIN.get(domain)
    m, e = run_hierarchical_domain(
        ddf,
        domain=domain,
        model_factory=lambda: make_base_model(BASE_MODEL_KIND),
        rare_threshold_l2=rare_thr_l2,
        min_local_support=200 if domain != "Moscow" else 100,
    )
    hier_rows.append(m)
    hier_errors.append(e)
    gc.collect()

hier_metrics = pd.concat(hier_rows, ignore_index=True)
hier_errors = pd.concat(hier_errors, ignore_index=True) if hier_errors else pd.DataFrame()

display(hier_metrics.sort_values(["domain", "label", "split", "macro_f1"], ascending=[True, True, True, False]))


,macro_f1,weighted_f1,accuracy,n_rows,arch,model,domain,label,split,classes_train,rare_labels_merged,local_l2_models
2,0.886294,0.990923,0.988327,1542,hierarchical,linear_char,Moscow,l1,test,10,0,7
0,0.994340,0.998215,0.998221,2811,hierarchical,linear_char,Moscow,l1,val,10,0,7
3,0.936297,0.981708,0.982490,1542,hierarchical,linear_char,Moscow,l2,test,24,8,7
1,0.994475,0.995445,0.995375,2811,hierarchical,linear_char,Moscow,l2,val,24,8,7
6,0.666363,0.900051,0.905374,173145,hierarchical,linear_char,Regions,l1,test,10,0,6
4,0.791688,0.756565,0.774263,27944,hierarchical,linear_char,Regions,l1,val,10,0,6
7,0.616637,0.912571,0.895775,173145,hierarchical,linear_char,Regions,l2,test,20,17,6
5,0.660940,0.812195,0.773368,27944,hierarchical,linear_char,Regions,l2,val,20,17,6


## 3) Router + domain heads (`l2`)

Router predicts domain, then corresponding head predicts `l2`.


In [12]:
def run_router_heads(df_map: Dict[str, pd.DataFrame], model_factory, rare_threshold_by_domain=None):
    parts = []
    for domain, ddf in df_map.items():
        tmp = filter_non_empty(ddf, "l2").copy()
        tmp["domain"] = domain
        parts.append(tmp)

    all_df = pd.concat(parts, ignore_index=True)
    X = build_text_features(all_df)
    y_domain = all_df["domain"].astype(str)
    groups = all_df["Source_File"].astype(str)

    tr_idx, va_idx, te_idx = split_group_holdout(X, y_domain, groups)
    X_tr, X_te = X.loc[tr_idx], X.loc[te_idx]
    d_tr, d_te = y_domain.loc[tr_idx], y_domain.loc[te_idx]

    router = model_factory()
    router.fit(X_tr, d_tr)
    d_pred = pd.Series(router.predict(X_te), index=X_te.index)

    # Global fallback l2 head
    y2_all_tr = clean_label(all_df.loc[tr_idx], "l2")
    global_head = model_factory()
    global_head.fit(X_tr, y2_all_tr)

    # Domain heads + rare mapping fitted on train part of each domain
    heads = {}
    mappers = {}
    for domain in sorted(all_df["domain"].unique()):
        idx = tr_idx.intersection(all_df.index[all_df["domain"] == domain])
        if len(idx) < 100:
            continue

        y_local_raw = clean_label(all_df.loc[idx], "l2")
        rare_thr = None if rare_threshold_by_domain is None else rare_threshold_by_domain.get(domain)
        mapper = RareLabelMapper(threshold=rare_thr)
        y_local = mapper.fit_transform(y_local_raw)

        if y_local.nunique() < 2:
            continue

        mdl = model_factory()
        mdl.fit(X.loc[idx], y_local)
        heads[domain] = mdl
        mappers[domain] = mapper

    # Predict l2 in batches by predicted domain
    y2_true_raw = clean_label(all_df.loc[te_idx], "l2")
    y2_pred = pd.Series(index=te_idx, dtype=object)

    # Start with global fallback
    y2_pred.loc[:] = global_head.predict(X.loc[te_idx])

    for domain_hat, idx in d_pred.groupby(d_pred).groups.items():
        idx = pd.Index(idx)
        mdl = heads.get(domain_hat)
        mapper = mappers.get(domain_hat)
        if mdl is None:
            continue
        pred_local = pd.Series(mdl.predict(X.loc[idx]), index=idx)

        # Ensure labels are in mapped space for fair eval
        if mapper is not None and mapper.rare_labels_:
            pass
        y2_pred.loc[idx] = pred_local.values

    # True labels transformed according to *true* domain mapper
    y2_true = y2_true_raw.copy()
    for domain, mapper in mappers.items():
        idx = te_idx.intersection(all_df.index[all_df["domain"] == domain])
        y2_true.loc[idx] = mapper.transform(y2_true.loc[idx]).values

    metrics = {
        "arch": "router_heads",
        "model": BASE_MODEL_KIND,
        "domain": "ALL",
        "label": "l2",
        "split": "test",
        "router_acc": accuracy_score(d_te, d_pred),
        "downstream_macro_f1": f1_score(y2_true, y2_pred, average="macro", zero_division=0),
        "downstream_weighted_f1": f1_score(y2_true, y2_pred, average="weighted", zero_division=0),
        "downstream_accuracy": accuracy_score(y2_true, y2_pred),
        "n_rows": len(te_idx),
        "n_heads": len(heads),
    }

    err = all_df.loc[te_idx, ["domain", "Source_File", "Layer", "Type", "Text", "l1", "l2"]].copy()
    err["arch"] = "router_heads"
    err["model"] = BASE_MODEL_KIND
    err["label"] = "l2"
    err["domain_true"] = d_te.values
    err["domain_pred"] = d_pred.values
    err["domain_error"] = err["domain_true"] != err["domain_pred"]
    err["y_true"] = y2_true.values
    err["y_pred"] = y2_pred.values
    err["is_error"] = err["y_true"] != err["y_pred"]

    return pd.DataFrame([metrics]), err


In [13]:
router_metrics, router_errors = run_router_heads(
    datasets,
    model_factory=lambda: make_base_model(BASE_MODEL_KIND),
    rare_threshold_by_domain=RARE_THRESHOLD_L2_BY_DOMAIN,
)

display(router_metrics)


,arch,model,domain,label,split,router_acc,downstream_macro_f1,downstream_weighted_f1,downstream_accuracy,n_rows,n_heads
0,router_heads,linear_char,ALL,l2,test,0.753515,0.493729,0.580098,0.604436,60458,2


## 4) Compare architectures


In [14]:
compare_df = pd.concat([
    single_metrics,
    hier_metrics,
], ignore_index=True)

compare_test_l2 = compare_df[(compare_df["split"] == "test") & (compare_df["label"] == "l2")].copy()
for c in ["macro_f1", "weighted_f1", "accuracy"]:
    compare_test_l2[c] = compare_test_l2[c].round(4)

display(compare_test_l2.sort_values(["domain", "arch", "macro_f1"], ascending=[True, True, False]))

display(router_metrics)


,macro_f1,weighted_f1,accuracy,n_rows,arch,model,domain,label,split,classes_train,rare_labels_merged,local_l2_models
11,0.9363,0.9817,0.9825,1542,hierarchical,linear_char,Moscow,l2,test,24,8,7.0
3,0.9391,0.9882,0.9890,1542,single_head,linear_char,Moscow,l2,test,24,8,NaN
15,0.6166,0.9126,0.8958,173145,hierarchical,linear_char,Regions,l2,test,20,17,6.0
7,0.6575,0.9217,0.9075,173145,single_head,linear_char,Regions,l2,test,20,17,NaN


,arch,model,domain,label,split,router_acc,downstream_macro_f1,downstream_weighted_f1,downstream_accuracy,n_rows,n_heads
0,router_heads,linear_char,ALL,l2,test,0.753515,0.493729,0.580098,0.604436,60458,2


## 5) Error analysis: inspect wrong predictions



In [15]:
all_errors = pd.concat([
    single_errors,
    hier_errors,
    router_errors,
], ignore_index=True)

mis = all_errors[all_errors["is_error"] == True].copy()
print(f"Total rows in error table: {len(all_errors):,}")
print(f"Misclassified rows: {len(mis):,}")

# Quick view
sample_cols = ["arch", "model", "domain", "label", "Source_File", "Layer", "Type", "Text", "y_true", "y_pred"]
display(mis[sample_cols].head(30))


Total rows in error table: 584,992
Misclassified rows: 74,437


,arch,model,domain,label,Source_File,Layer,Type,Text,y_true,y_pred
785,single_head,linear_char,Moscow,l1,output(1-2)_3_ДЖКХпр-24_00875up.dxf,Кабель электрический,TEXT,176.77в.тр.,Инженерные сети,Рельеф
786,single_head,linear_char,Moscow,l1,output(1-2)_3_ДЖКХпр-24_00875up.dxf,Кабель электрический,TEXT,176.69в.тр.,Инженерные сети,Рельеф
787,single_head,linear_char,Moscow,l1,output(1-2)_3_ДЖКХпр-24_00875up.dxf,Кабель электрический,TEXT,176.98в.тр.,Инженерные сети,Рельеф
793,single_head,linear_char,Moscow,l1,output(1-2)_3_ДЖКХпр-24_00875up.dxf,Кабель электрический,TEXT,176.34в.тр.,Инженерные сети,Рельеф
794,single_head,linear_char,Moscow,l1,output(1-2)_3_ДЖКХпр-24_00875up.dxf,Кабель электрический,TEXT,176.17в.тр.,Инженерные сети,Рельеф
795,single_head,linear_char,Moscow,l1,output(1-2)_3_ДЖКХпр-24_00875up.dxf,Кабель электрический,TEXT,176.47в.тр.,Инженерные сети,Рельеф
871,single_head,linear_char,Moscow,l1,output(1-2)_3_ДЖКХпр-24_00875up.dxf,Водосток,TEXT,177.23лот.,Инженерные сети,Служебные элементы
873,single_head,linear_char,Moscow,l1,output(1-2)_3_ДЖКХпр-24_00875up.dxf,Водосток,TEXT,176.94лот.,Инженерные сети,Служебные элементы
874,single_head,linear_char,Moscow,l1,output(1-2)_3_ДЖКХпр-24_00875up.dxf,Водосток,TEXT,176.94лот.,Инженерные сети,Служебные элементы
875,single_head,linear_char,Moscow,l1,output(1-2)_3_ДЖКХпр-24_00875up.dxf,Водосток,TEXT,176.72лот.,Инженерные сети,Служебные элементы


In [34]:
for domain in ["Moscow", "Regions"]:
    df = datasets[domain][["Layer", "Type", "Text", "l1", "l2"]].copy()
    df = df.dropna(subset=["l1", "l2"])
    df = df[df["l1"].str.strip() != ""]
    df = df[df["l2"].str.strip() != ""]
    df = df.drop_duplicates(subset=["Layer", "Type", "Text"])
    df = df.sort_values(["l1", "l2", "Layer"])

    path = OUT_DIR / f"unique_labels_{domain}.xlsx"
    df.to_excel(path, index=False, engine="openpyxl")
    print(f"{domain}: {len(df):,} уникальных записей -> {path}")


Moscow: 8,357 уникальных записей -> C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\geo\reports\arch_tests\unique_labels_Moscow.xlsx
Regions: 51,517 уникальных записей -> C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\geo\reports\arch_tests\unique_labels_Regions.xlsx


In [ ]:
import re
import pandas as pd
from pathlib import Path


TEXT_TYPES = {"TEXT", "MTEXT", "ТЕКСТ", "МТЕКСТ"}


RE_INT_1_4       = re.compile(r"^\d{1,4}$")
RE_INT_1_8       = re.compile(r"^\d{1,8}$")
RE_FLOAT         = re.compile(r"^-?\d+\.\d+$")
RE_FLOAT_ANY_SEP = re.compile(r"^-?\d+[.,]\d+$")
RE_HEIGHT        = re.compile(r"^H:\s*-?\d+(?:\.\d+)?m$")
RE_POSITION      = re.compile(r"^E:\s*-?\d+(?:\.\d+)?m\s+N:\s*-?\d+(?:\.\d+)?m$")
RE_VT            = re.compile(r"^vt\d{1,5}$")
RE_DG            = re.compile(r"^dg\d{1,5}$")
RE_VD            = re.compile(r"^vd\d{1,5}$")
RE_CADASTRAL     = re.compile(r"^\d+(?:[:\-.]\d+)+(?:\([^)]+\))*$")
RE_HP_FLOAT      = re.compile(r"^\\P\s*-?\d+(?:\.\d+)?$")


RE_NUMBER        = re.compile(r"^\d+$|^-?\d+[.,]\d+$")
RE_HEIGHT_OR_FLOAT = re.compile(r"^(?:H:\s*-?\d+(?:\.\d+)?m|-?\d+\.\d+)$")
RE_INT_OR_FLOAT    = re.compile(r"^(?:\d{1,8}|-?\d+\.\d+)$")
RE_PAREN_TOKEN     = re.compile(r"^\([^)]+\)$")


DEDUPE_RULES: dict[str, tuple[re.Pattern, str]] = {

    "POINTS":        (RE_INT_1_4,         "<N_1_4>"),
    "откосы":        (RE_FLOAT,           "<FLOAT>"),
    "LABELS":        (RE_CADASTRAL,       "<CADASTRAL>"),
    "Координатная":  (RE_INT_1_8,         "<N_1_8>"),
    "Сетка":         (RE_INT_1_8,         "<N_1_8>"),
    "00p":           (RE_INT_1_8,         "<N_1_8>"),
    "00r":           (RE_INT_1_8,         "<N_1_8>"),
    "013":           (RE_INT_1_8,         "<N_1_8>"),
    "11b":           (RE_INT_OR_FLOAT,    "<INT_OR_FLOAT>"),
    "Heights":       (RE_HEIGHT_OR_FLOAT, "<H_or_FLOAT>"),
    "Name_1":        (RE_VT,              "<vtN>"),
    "Name_2":        (RE_DG,              "<dgN>"),
    "Name_3":        (RE_VD,              "<vdN>"),
    "Positions":     (RE_POSITION,        "<E_N_m>"),
    "Откос":         (RE_FLOAT,           "<FLOAT>"),
    "Дороги":        (RE_FLOAT,           "<FLOAT>"),


    "3номер дерева":     (RE_NUMBER, "<NUM>"),
    "НОМЕР":             (RE_NUMBER, "<NUM>"),
    "НОМЕРА ДЕРЕВЬЕВ":   (RE_NUMBER, "<NUM>"),
    "Номер дерева":      (RE_NUMBER, "<NUM>"),
    "архНомер":          (RE_NUMBER, "<NUM>"),
    "дерНомер":          (RE_NUMBER, "<NUM>"),
    "номер деревьев":    (RE_NUMBER, "<NUM>"),
    "номера деревьев":   (RE_NUMBER, "<NUM>"),
    "топоНомер":         (RE_NUMBER, "<NUM>"),
    "PI_NUDEFAULT":      (RE_NUMBER, "<NUM>"),

    "PI_OTDEFAULT":                (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "Высоты":                      (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "Доп.отметки":                 (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "Отметки высот":               (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "Рельеф":                      (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "Рельеф__грунты_и_микроформы": (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "архОтметка":                  (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "высоты деревьев":             (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "дерОтметка":                  (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "отметка":                     (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "отметки":                     (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "топоОтметка":                 (RE_FLOAT_ANY_SEP, "<FLOAT>"),

    "Survey.td2_Mark":             (RE_NUMBER,       "<NUM>"),
    "Survey.td2_point_desc_Mark":  (RE_PAREN_TOKEN,  "<(TOKEN)>"),

    "Гидрография":            (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "гидрография":            (RE_FLOAT_ANY_SEP, "<FLOAT>"),
    "Тропы":                  (RE_NUMBER,        "<NUM>"),
    "Территория Лиры":        (RE_NUMBER,        "<NUM>"),
    "Height_1":               (RE_HP_FLOAT,      "<P_FLOAT>"),
    "подпись горизонталей":   (RE_INT_1_8,       "<N_1_8>"),
    "Горизонтали--LBL":       (RE_INT_1_8,       "<N_1_8>"),
    "Основные горизонтали":   (RE_INT_1_8,       "<N_1_8>"),

    "дороги":           (RE_FLOAT,    "<FLOAT>"),
    "сетка":            (RE_INT_1_8,  "<N_1_8>"),
    "Сетка координат":  (RE_INT_1_8,  "<N_1_8>"),
    "SETK_NAD":         (RE_INT_1_8,  "<N_1_8>"),
    "DEFAULT":          (RE_INT_1_8,  "<N_1_8>"),
}


KEEP_SEPARATE_LAYERS = {""}


def normalize_row(layer, type_, text):
    if not isinstance(text, str):
        return text
    if type_ not in TEXT_TYPES:
        return text
    if layer in KEEP_SEPARATE_LAYERS:
        return text
    rule = DEDUPE_RULES.get(layer)
    if rule is None:
        return text
    regex, placeholder = rule
    return placeholder if regex.fullmatch(text.strip()) else text


domain = "Regions"

df = datasets[domain][["Layer", "Type", "Text", "l1", "l2"]].copy()
df = df.dropna(subset=["l1", "l2"])
df = df[df["l1"].astype(str).str.strip() != ""]
df = df[df["l2"].astype(str).str.strip() != ""]

before = len(df)

df["_key"] = [
    normalize_row(str(l), str(t), x)
    for l, t, x in zip(df["Layer"], df["Type"], df["Text"])
]

counts = (
    df.groupby(["Layer", "Type", "_key"], dropna=False)
      .size().rename("_count").reset_index()
)

df_dedup = (
    df.drop_duplicates(subset=["Layer", "Type", "_key"])
      .merge(counts, on=["Layer", "Type", "_key"], how="left")
      .drop(columns=["_key"])
      .sort_values(["l1", "l2", "Layer"])
)

path = OUT_DIR / f"unique_labels_{domain}.xlsx"
df_dedup.to_excel(path, index=False, engine="openpyxl")
print(f"{domain}: было {before:,} -> стало {len(df_dedup):,} строк "
      f"(схлопнуто {before - len(df_dedup):,}) -> {path}")


diag = (
    df_dedup[df_dedup["Type"].isin(TEXT_TYPES)]
      .groupby("Layer").size()
      .sort_values(ascending=False).head(30)
)
print("\nТоп-30 слоёв по количеству TEXT/MTEXT строк после дедупа:")
print(diag.to_string())

Regions: было 384,457 -> стало 2,732 строк (схлопнуто 381,725) -> C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\geo\reports\arch_tests\unique_labels_Regions.xlsx

Топ-30 слоёв по количеству TEXT/MTEXT строк после дедупа:
Layer
12492                     212
0                         173
канализация ливневая      142
Топонимика                140
канализация               122
11b                       110
топоОтметка               103
Code_2                     96
Code_1                     93
12493                      78
00r                        70
границы зданий             61
CABEL                      52
Водопровод                 46
Автодорожное хозяйство     45
Кабели связи               38
SVAZ                       37
Высоты                     36
Водослив                   36
дороги                     33
Рельеф                     32
Горизонтали                32
POINTS                     29
00p                        28
Code_3                     27
отмет

In [ ]:

df_files = datasets["Regions"][["Layer", "Type", "Text", "Source_File", "l1", "l2"]].copy()
df_files = df_files.dropna(subset=["l1", "l2"])
df_files = df_files[(df_files["l1"].str.strip() != "") & (df_files["l2"].str.strip() != "")]
df_files["_text_norm"] = df_files.apply(lambda r: normalize_text_regions(r["Layer"], r["Text"]), axis=1)
df_files = df_files.drop_duplicates(subset=["Layer", "Type", "_text_norm", "Source_File"])

df_files = (
    df_files.groupby(["Layer", "Type", "_text_norm", "l1", "l2"], as_index=False)
    .agg(Source_Files=("Source_File", lambda x: ", ".join(sorted(x.unique()))))
)
df_files = df_files.rename(columns={"_text_norm": "Text_pattern"}).sort_values(["l1", "l2", "Layer"])

df_files.to_excel(OUT_DIR / "unique_labels_Regions_by_file.xlsx", index=False, engine="openpyxl")
print(f"Regions by file: {len(df_files):,} rows")


Regions by file: 30,738 rows


In [39]:
df_msk_files = datasets["Moscow"][["Layer", "Type", "Text", "Source_File", "l1", "l2"]].copy()
df_msk_files = df_msk_files.dropna(subset=["l1", "l2"])
df_msk_files = df_msk_files[(df_msk_files["l1"].str.strip() != "") & (df_msk_files["l2"].str.strip() != "")]
df_msk_files = df_msk_files.drop_duplicates(subset=["Layer", "Type", "Text", "Source_File"])

df_msk_files = (
    df_msk_files.groupby(["Layer", "Type", "Text", "l1", "l2"], as_index=False)
    .agg(Source_Files=("Source_File", lambda x: ", ".join(sorted(x.unique()))))
)
df_msk_files = df_msk_files.sort_values(["l1", "l2", "Layer"])

df_msk_files.to_excel(OUT_DIR / "unique_labels_Moscow_by_file.xlsx", index=False, engine="openpyxl")
print(f"Moscow by file: {len(df_msk_files):,} rows")


Moscow by file: 8,357 rows


In [40]:
maf_unknown = []

for domain in ["Moscow", "Regions"]:
    df = datasets[domain][["Layer", "Type", "Text", "Source_File", "l1", "l2"]].copy()
    df["domain"] = domain
    df["l2_str"] = df["l2"].fillna("").astype(str).str.strip()
    
    mask = (
        (df["l1"].astype(str).str.strip() == "МАФ") &
        (df["l2_str"].isin(["", "None", "NaN", "nan", "Неизвестно"]))
    )
    maf_unknown.append(df[mask])

maf_df = pd.concat(maf_unknown, ignore_index=True).drop(columns=["l2_str"])
maf_df = maf_df.drop_duplicates(subset=["domain", "Layer", "Type", "Text"])
maf_df = maf_df.sort_values(["domain", "Layer"])

maf_df.to_excel(OUT_DIR / "maf_unknown_l2.xlsx", index=False, engine="openpyxl")
print(f"МАФ -> Неизвестно/None: {len(maf_df):,} rows")
display(maf_df.groupby(["domain", "Layer"])["Text"].count().reset_index().rename(columns={"Text": "count"}))


МАФ -> Неизвестно/None: 15 rows


,domain,Layer,count
0,Regions,МАФ,15


In [42]:
result = []

for domain in ["Moscow", "Regions"]:
    df = datasets[domain][["Layer", "Type", "Text", "Source_File", "l1", "l2"]].copy()
    df["domain"] = domain

    mask = (
        (df["l1"].astype(str).str.strip() == "Служебные элементы") &
        (df["l2"].astype(str).str.strip() == "Точки")
    )
    result.append(df[mask])

result_df = pd.concat(result, ignore_index=True)
result_df = result_df.drop_duplicates(subset=["domain", "Layer", "Type", "Text"])
result_df = result_df.sort_values(["domain", "Layer"])

result_df.to_excel(OUT_DIR / "служебные_точки.xlsx", index=False, engine="openpyxl")
print(f"Всего: {len(result_df):,} rows")
display(result_df.groupby(["domain", "Layer"])["Text"].count().reset_index().rename(columns={"Text": "count"}))


Всего: 48 rows


,domain,Layer,count
0,Regions,0,1
1,Regions,00p,1
2,Regions,1,1
3,Regions,12330,1
4,Regions,2пикет,1
5,Regions,Height,1
6,Regions,PI_STDEFAULT,1
7,Regions,PI_TT0,1
8,Regions,PI_TTDEFAULT,1
9,Regions,SVAZ,1


In [43]:
result = []

for domain in ["Moscow", "Regions"]:
    df = datasets[domain][["Layer", "Type", "Text", "Source_File", "l1", "l2"]].copy()
    df["domain"] = domain

    mask = (
        (df["Layer"].astype(str).str.strip() == "Леса и газоны") &
        (df["Text"].astype(str).str.strip() == "лист")
    )
    result.append(df[mask])

result_df = pd.concat(result, ignore_index=True)
result_df = result_df.drop_duplicates(subset=["domain", "Layer", "Type", "Text", "Source_File"])
result_df = result_df.sort_values(["domain", "Source_File"])

result_df.to_excel(OUT_DIR / "леса_лист.xlsx", index=False, engine="openpyxl")
print(f"Всего: {len(result_df):,} rows")
display(result_df)


Всего: 0 rows


,Layer,Type,Text,Source_File,l1,l2,domain


In [16]:
# Top confusions for focused labeling / rule improvements
for (arch, domain, label), grp in mis.groupby(["arch", "domain", "label"]):
    print(f"\n=== {arch} | {domain} | {label} ===")
    top = summarize_confusions(grp["y_true"], grp["y_pred"], top_k=15)
    display(top)



=== hierarchical | Moscow | l2 ===


,y_true,y_pred,count
1,Координаты,unknown,14
2,Ливневая канализация,Координаты,10
0,Газон,Деревья,3



=== hierarchical | Regions | l2 ===


,y_true,y_pred,count
26,Точки,Номер дерева,12029
24,Точки,Контуры,1627
17,Отметки высот,Неизвестно,1548
1,Асфальт,Контуры,1272
11,Неизвестно,Контуры,333
25,Точки,Неизвестно,231
27,Точки,Электроснабжение,186
23,Точки,unknown,179
14,Неизвестно,Текст,167
19,Текст,Газон,128



=== router_heads | Moscow | l2 ===


...

,y_true,y_pred,count
1,Координаты,unknown,14
0,Газон,Деревья,3



=== single_head | Regions | l1 ===


,y_true,y_pred,count
18,Служебные элементы,Озеленение,12157
13,Рельеф,Неизвестно,1619
21,Твердые покрытия,Служебные элементы,1290
2,Здания и сооружения,Служебные элементы,475
15,Служебные элементы,Инженерные сети,378
16,Служебные элементы,МАФ,186
19,Служебные элементы,Рельеф,85
17,Служебные элементы,Неизвестно,58
14,Служебные элементы,Водные объекты,47
10,Неизвестно,Служебные элементы,36



=== single_head | Regions | l2 ===


,y_true,y_pred,count
26,Точки,Номер дерева,12029
17,Отметки высот,Неизвестно,1548
24,Точки,Газон,758
3,Асфальт,Отметки высот,528
27,Точки,Электроснабжение,186
23,Точки,unknown,179
15,Неизвестно,Текст,179
4,Асфальт,Плитка,141
19,Текст,Газон,128
22,Текст,Отметки высот,84


In [ ]:
FILTER_ARCH = None
FILTER_DOMAIN = None
FILTER_LABEL = "l2"
FILTER_TRUE = None
FILTER_PRED = None

export_df = mis.copy()
if FILTER_ARCH is not None:
    export_df = export_df[export_df["arch"] == FILTER_ARCH]
if FILTER_DOMAIN is not None:
    export_df = export_df[export_df["domain"] == FILTER_DOMAIN]
if FILTER_LABEL is not None:
    export_df = export_df[export_df["label"] == FILTER_LABEL]
if FILTER_TRUE is not None:
    export_df = export_df[export_df["y_true"] == FILTER_TRUE]
if FILTER_PRED is not None:
    export_df = export_df[export_df["y_pred"] == FILTER_PRED]

export_cols = [
    "arch", "model", "domain", "label", "Source_File", "Layer", "Type", "Text",
    "y_true", "y_pred", "is_error", "margin",
    "domain_true", "domain_pred", "domain_error", "is_l1_error",
]
export_cols = [c for c in export_cols if c in export_df.columns]

path = export_errors(export_df[export_cols], name="misclassified")
print(path)


Saved: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\geo\reports\arch_tests\misclassified_errors_20260419_112431.csv
C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\geo\reports\arch_tests\misclassified_errors_20260419_112431.csv


In [23]:
export_cols = [
    "label", "Layer", "Type", "Text","y_true", "y_pred",
]
export_cols = [c for c in export_cols if c in export_df.columns]

ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
path = OUT_DIR / f"misclassified_errors_{ts}.xlsx"
export_df[export_cols].drop_duplicates(subset=["Layer", "Type", "Text", "y_true", "y_pred"]).to_excel(path, index=False, engine="openpyxl")
print(f"Saved: {path}")


Saved: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\geo\reports\arch_tests\misclassified_errors_20260419_114305.xlsx


In [30]:
import re

FILTER_ARCH = "single_head"   # "single_head" | "hierarchical" | "router_heads"
FILTER_DOMAIN = None

base = mis[mis["arch"] == FILTER_ARCH].copy()
if FILTER_DOMAIN:
    base = base[base["domain"] == FILTER_DOMAIN]

export_df = base[base["label"] == "l2"].copy()
export_df = export_df.rename(columns={
    "y_true": "l2_true",
    "y_pred": "l2_pred",
    "is_error": "l2_error",
})

if "y1_pred" in export_df.columns:
    export_df = export_df.rename(columns={
        "y1_true": "l1_true_hier",
        "y1_pred": "l1_pred_hier",
        "is_l1_error": "l1_error",
    })

export_cols = ["arch", "model", "domain", "Source_File", "Layer", "Type", "Text",
               "l1", "l2_true", "l2_pred", "l2_error",
               "l1_pred_hier", "l1_error"]
export_cols = [c for c in export_cols if c in export_df.columns]

def normalize_for_dedup(text):
    return re.sub(r"\d{3}\.\d{2}", "<NUM>", str(text))

export_df = export_df[export_cols].copy()
export_df["_text_norm"] = export_df["Text"].apply(normalize_for_dedup)
export_df = export_df.drop_duplicates(subset=["Layer", "Type", "_text_norm", "l1", "l2_true", "l2_pred"])
export_df = export_df.drop(columns=["_text_norm"])

ts = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
path = OUT_DIR / f"misclassified_errors_{ts}.xlsx"
export_df.to_excel(path, index=False, engine="openpyxl")
print(f"Saved: {path} | rows={len(export_df):,}")


Saved: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\geo\reports\arch_tests\misclassified_errors_20260419_171514.xlsx | rows=182
